In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
import warnings
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Kutuphaneler yuklendi.")

In [ ]:
# --- KONFIGURASYON (ResNeXt50 + CutMix + Rotation Aug) ---
MODEL_NAME = 'resnext50_32x4d'
EXPERIMENT_NAME = "ResNeXt50_CutMixAug_R1"

# Hiperparametreler (en iyi ablation run'indan alinmistir)
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.0003
NUM_CLASSES = 8
DROPOUT_RATE = 0.3
WEIGHT_DECAY = 0.0001
LABEL_SMOOTHING = 0.05
INPUT_SIZE = 224
USE_CLASS_WEIGHTED_LOSS = True
FORCE_WEIGHTED_SAMPLER = False
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_METRIC = 'val_macro_f1'
NUM_WORKERS = 2 if os.name == 'nt' else 4

# CutMix parametreleri
CUTMIX_ALPHA = 1.0
CUTMIX_PROB  = 0.5

# Run etiketleri
dropout_tag  = f"dropout{int(round(DROPOUT_RATE * 10)):02d}"
ls_tag       = f"ls{int(round(LABEL_SMOOTHING * 100)):02d}"
wd_tag       = f"wd{str(WEIGHT_DECAY).replace('.', 'p').replace('-', 'm')}"
lr_tag       = f"lr{LEARNING_RATE:g}".replace(".", "p")
input_tag    = f"i{INPUT_SIZE}"
sampler_tag  = "smpON" if FORCE_WEIGHTED_SAMPLER else "smpOFF"
cwl_tag      = "cwlON" if USE_CLASS_WEIGHTED_LOSS else "cwlOFF"
RUN_TAG  = f"{dropout_tag}_{ls_tag}_{wd_tag}_bs{BATCH_SIZE}_ep{EPOCHS}_{lr_tag}_{input_tag}_cutmixAug_{sampler_tag}_{cwl_tag}"
RUN_NAME = f"{EXPERIMENT_NAME}_{RUN_TAG}"

cwd = os.getcwd()
if os.path.isdir(os.path.join(cwd, "data", "prepared-data")):
    PROJECT_ROOT = cwd
elif os.path.isdir(os.path.join(cwd, "..", "data", "prepared-data")):
    PROJECT_ROOT = os.path.abspath(os.path.join(cwd, ".."))
else:
    PROJECT_ROOT = cwd

DATA_DIR          = os.path.join(PROJECT_ROOT, "data", "prepared-data")
OUTPUT_DIR        = os.path.join(PROJECT_ROOT, "models", "pytorch", RUN_NAME)
RESULT_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs", MODEL_NAME)
PLOTS_DIR         = os.path.join(RESULT_OUTPUT_DIR, "plots", "cutmix_aug")
REPORTS_DIR       = os.path.join(RESULT_OUTPUT_DIR, "reports")
BEST_MODEL_PATH   = os.path.join(OUTPUT_DIR, f"best_{MODEL_NAME}.pth")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Run Tag: {RUN_TAG}")
print(f"CutMix Alpha: {CUTMIX_ALPHA}, Prob: {CUTMIX_PROB}")

In [ ]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.15),
        transforms.RandomRotation(degrees=15),
        transforms.RandomApply([
            transforms.ColorJitter(brightness=0.1, contrast=0.12, saturation=0.08, hue=0.02)
        ], p=0.4),
        transforms.RandomAutocontrast(p=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ['train', 'val', 'test']}

train_targets  = np.array(image_datasets['train'].targets)
class_counts   = np.bincount(train_targets, minlength=NUM_CLASSES)
class_weights  = class_counts.sum() / (NUM_CLASSES * np.maximum(class_counts, 1))
CLASS_WEIGHTS_TENSOR = torch.tensor(class_weights, dtype=torch.float32)

train_sampler = None
if FORCE_WEIGHTED_SAMPLER:
    sample_weights = class_weights[train_targets]
    train_sampler  = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights), replacement=True
    )

loader_common_kwargs = {'num_workers': NUM_WORKERS, 'pin_memory': (DEVICE == 'cuda')}
if NUM_WORKERS > 0:
    loader_common_kwargs['persistent_workers'] = True
    loader_common_kwargs['prefetch_factor'] = 2

dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE,
                        shuffle=(train_sampler is None), sampler=train_sampler, **loader_common_kwargs),
    'val':   DataLoader(image_datasets['val'],   batch_size=BATCH_SIZE, shuffle=False, **loader_common_kwargs),
    'test':  DataLoader(image_datasets['test'],  batch_size=BATCH_SIZE, shuffle=False, **loader_common_kwargs)
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names   = image_datasets['train'].classes
print(f"Siniflar: {class_names}")
print(f"Egitim: {dataset_sizes['train']}  Val: {dataset_sizes['val']}  Test: {dataset_sizes['test']}")

In [ ]:
def cutmix_data(inputs, targets, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    batch_size = inputs.size(0)
    rand_index = torch.randperm(batch_size, device=inputs.device)
    targets_a, targets_b = targets, targets[rand_index]

    _, _, H, W = inputs.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    inputs[:, :, bby1:bby2, bbx1:bbx2] = inputs[rand_index, :, bby1:bby2, bbx1:bbx2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1)) / (W * H)
    return inputs, targets_a, targets_b, lam

def cutmix_criterion(criterion, outputs, targets_a, targets_b, lam):
    return lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)

print("CutMix fonksiyonlari tanimlandi.")

In [ ]:
def create_model():
    print(f"Model indiriliyor: {MODEL_NAME}...")
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, drop_rate=DROPOUT_RATE)
    return model

model = create_model().to(DEVICE)

loss_weight = CLASS_WEIGHTS_TENSOR.to(DEVICE) if USE_CLASS_WEIGHTED_LOSS else None
criterion   = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, weight=loss_weight)
optimizer   = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler   = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.3, patience=4, verbose=True)
print(f"Parametre sayisi: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    since = time.time()
    best_acc = best_val_macro_f1 = 0.0
    best_val_loss = float('inf')
    best_metric_value = -float('inf')
    epochs_without_improvement = 0
    history = {
        'train_loss': [], 'train_acc': [], 'train_macro_f1': [],
        'val_loss':   [], 'val_acc':   [], 'val_macro_f1':   [], 'lr': []
    }

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss = running_corrects = 0
            phase_labels, phase_preds = [], []

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    use_cutmix = (phase == 'train') and (np.random.rand() < CUTMIX_PROB)
                    if use_cutmix:
                        inputs, ta, tb, lam = cutmix_data(inputs, labels, CUTMIX_ALPHA)
                        outputs = model(inputs)
                        loss    = cutmix_criterion(criterion, outputs, ta, tb, lam)
                    else:
                        outputs = model(inputs)
                        loss    = criterion(outputs, labels)

                    _, preds = torch.max(outputs, 1)
                    if phase == 'train':
                        loss.backward(); optimizer.step()

                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                phase_labels.extend(labels.detach().cpu().numpy())
                phase_preds.extend(preds.detach().cpu().numpy())

            epoch_loss     = running_loss / dataset_sizes[phase]
            epoch_acc      = running_corrects.double() / dataset_sizes[phase]
            epoch_macro_f1 = f1_score(phase_labels, phase_preds, average='macro')
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} MacroF1: {epoch_macro_f1:.4f}')

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())
            history[f'{phase}_macro_f1'].append(epoch_macro_f1)

            if phase == 'val':
                scheduler.step(epoch_macro_f1)
                current_lr = optimizer.param_groups[0]['lr']
                history['lr'].append(current_lr)
                if epoch_acc > best_acc: best_acc = epoch_acc
                if epoch_macro_f1 > best_val_macro_f1: best_val_macro_f1 = epoch_macro_f1
                if epoch_loss < best_val_loss: best_val_loss = epoch_loss

                if epoch_macro_f1 > best_metric_value:
                    best_metric_value = epoch_macro_f1
                    epochs_without_improvement = 0
                    with open(BEST_MODEL_PATH, 'wb') as f:
                        torch.save(model.state_dict(), f)
                    print(f"En Iyi Model (MacroF1: {best_metric_value:.4f}) -> kaydedildi.")
                else:
                    epochs_without_improvement += 1
                    print(f"F1 iyilesmedi: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Erken durdurma tetiklendi."); break

    elapsed = time.time() - since
    print(f'\nEgitim Tamamlandi: {elapsed // 60:.0f}dk {elapsed % 60:.0f}sn')
    print(f'En Iyi Val Acc: {best_acc:.4f}  En Iyi Val MacroF1: {best_val_macro_f1:.4f}')

    with open(BEST_MODEL_PATH, 'rb') as f:
        model.load_state_dict(torch.load(f, map_location=DEVICE))
    return model, history


model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=EPOCHS)

In [ ]:
plt.figure(figsize=(18, 5))
for i, (key_train, key_val, title, ylabel) in enumerate([
    ('train_acc',      'val_acc',      f'{MODEL_NAME} Accuracy (CutMix+Rot)',  'Accuracy'),
    ('train_loss',     'val_loss',     f'{MODEL_NAME} Loss (CutMix+Rot)',      'Loss'),
    ('train_macro_f1', 'val_macro_f1', f'{MODEL_NAME} Macro F1 (CutMix+Rot)', 'Macro F1')
]):
    plt.subplot(1, 3, i + 1)
    plt.plot(history.get(key_train, []), label='Train')
    plt.plot(history.get(key_val,   []), label='Val')
    plt.title(title); plt.xlabel('Epochs'); plt.ylabel(ylabel)
    plt.legend(); plt.grid(True)

plot_path = os.path.join(PLOTS_DIR, f'training_graph_{MODEL_NAME}_{RUN_TAG}.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Grafikler kaydedildi: {plot_path}")

In [ ]:
print("\nTEST SETI DEGERLENDIRMESI")
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        outputs  = model(inputs.to(DEVICE))
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

report   = classification_report(y_true, y_pred, target_names=class_names, digits=4)
macro_f1 = f1_score(y_true, y_pred, average='macro')
print(report)
print(f"Macro F1: {macro_f1:.4f}")

with open(os.path.join(REPORTS_DIR, f'classification_report_{MODEL_NAME}_{RUN_TAG}.txt'), 'w', encoding='utf-8') as f:
    f.write(report + f"\nMacro F1: {macro_f1:.6f}\n")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix - {MODEL_NAME} (CutMix+Rot)')
plt.xlabel('Tahmin Edilen'); plt.ylabel('Gercek')
cm_path = os.path.join(PLOTS_DIR, f'confusion_matrix_{MODEL_NAME}_{RUN_TAG}.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()

name_to_idx = {name: i for i, name in enumerate(class_names)}
pair_1_err = pair_2_err = None
if 'esophagitis' in name_to_idx and 'normal-z-line' in name_to_idx:
    i, j = name_to_idx['esophagitis'], name_to_idx['normal-z-line']
    pair_1_err = int(cm[i, j] + cm[j, i])
if 'dyed-lifted-polyps' in name_to_idx and 'dyed-resection-margins' in name_to_idx:
    i, j = name_to_idx['dyed-lifted-polyps'], name_to_idx['dyed-resection-margins']
    pair_2_err = int(cm[i, j] + cm[j, i])

latest_metrics = {
    'run_name': RUN_NAME, 'run_tag': RUN_TAG, 'model_name': MODEL_NAME,
    'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'dropout_rate': DROPOUT_RATE,
    'label_smoothing': LABEL_SMOOTHING, 'input_size': INPUT_SIZE,
    'cutmix_alpha': CUTMIX_ALPHA, 'cutmix_prob': CUTMIX_PROB,
    'force_weighted_sampler': FORCE_WEIGHTED_SAMPLER,
    'use_weighted_sampler': train_sampler is not None,
    'use_class_weighted_loss': USE_CLASS_WEIGHTED_LOSS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_metric': EARLY_STOPPING_METRIC,
    'best_val_acc': float(max(history['val_acc'])),
    'best_val_loss': float(min(history['val_loss'])),
    'best_val_macro_f1': float(max(history['val_macro_f1'])),
    'test_macro_f1': float(macro_f1),
    'pair_err_esophagitis_normal_z_line': pair_1_err,
    'pair_err_dyed_lifted_vs_resection': pair_2_err
}

ablation_csv_path = os.path.join(RESULT_OUTPUT_DIR, f'ablation_results_{MODEL_NAME}.csv')
new_row_df = pd.DataFrame([latest_metrics])
if os.path.exists(ablation_csv_path):
    old_df = pd.read_csv(ablation_csv_path)
    if 'run_tag' in old_df.columns:
        old_df = old_df[old_df['run_tag'] != RUN_TAG]
    result_df = pd.concat([old_df, new_row_df], ignore_index=True)
else:
    result_df = new_row_df
result_df.to_csv(ablation_csv_path, index=False)
print(f"Ablation sonuclari kaydedildi: {ablation_csv_path}")
print(new_row_df[['run_tag','best_val_acc','best_val_macro_f1','test_macro_f1']].to_string(index=False))